# Splat Studio 0.4.1 · Colab 使用笔记本

与同目录《完整使用说明.html》配套。先在 App 建立场景、选择云端并导出任务 ZIP。

**推荐第一次：Teatime 177 张 / 均衡 / 识别物品 / 重点“熊” / A100 / 允许首次下载资源。**

在 Colab 选择 GPU（A100），从第 1 步到第 7 步顺序运行。首次运行识别任务后，会停在需要你回 App 确认区域的位置。确认后导出新的重建任务，再从第 2 步运行到第 7 步。不要跳过确认，也不要手改任务 ZIP。

计算环境：独立虚拟环境、torch 2.11.0、torchvision 0.26.0、CUDA Toolkit 12.8、固定项目语义依赖。安装命令和模块导入检查会显示实际结果；本笔记本没有伪造预运行输出。

- 选择 A100 仍取决于你的 Colab 权限和资源供应；GPU 不符会明确停止。
- 识别阶段也需要 3DGS 扩展；完整 177 张的相机恢复需要 COLMAP。
- 本笔记本为 apt 安装的 COLMAP 指定 CPU 特征提取/匹配；高斯及语义优化使用 GPU。
- 允许首次安装会联网下载依赖与模型。可选 Drive 备份需要你授权。
- 训练执行在 Colab，App 没有远程实时进度。Colab 断开/额度耗尽不保证继续运行，worker 当前不支持自动断点续训。


## 1 · 初始化并检查 GPU

每次新连接 Colab 时运行一次。需要 Linux、Python 3.11–3.13 和 NVIDIA GPU；本次请在 Colab 选择 A100。


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, zipfile, hashlib, shutil, subprocess, stat, time, uuid, re, signal, threading
assert sys.platform == 'linux', '此笔记本在 Colab / Linux GPU 环境运行，不在 Mac 上运行。'
assert (3,11) <= sys.version_info[:2] <= (3,13), '需要 Python 3.11–3.13；建议使用 Python 3.12 环境。'
BASE = Path('/content/splat-studio-v041')
BASE.mkdir(parents=True, exist_ok=True)
ENV_DIR = BASE / 'env'
PY = ENV_DIR / 'bin/python'
DRIVE_ROOT = None

def digest(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def run(args, log=None, cwd=None):
    args = [str(x) for x in args]
    stream = open(log, 'w') if log else None
    proc = subprocess.Popen(args, cwd=cwd, env=os.environ.copy(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if stream: stream.write(line); stream.flush()
        if proc.wait(): raise RuntimeError(f'命令失败，日志：{log or "上方输出"}')
    except BaseException:
        # 只停止本单元格启动的进程及其训练子进程。
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        raise
    finally:
        if stream: stream.close()

def unpack_task(package, destination):
    """在执行包内代码前检查大小、路径和每个文件的 SHA256。"""
    package, destination = Path(package), Path(destination)
    if destination.exists(): raise ValueError('解压目录必须是新目录。')
    if package.stat().st_size > 4*1024**3: raise ValueError('任务包超过 4 GiB。')
    with zipfile.ZipFile(package) as z:
        infos=z.infolist(); seen=set(); total=0
        if len(infos)>20001: raise ValueError('任务文件过多。')
        for info in infos:
            name=info.filename; parts=name.split('/'); mode=info.external_attr>>16
            if (not name or len(name)>512 or any(c in name for c in ('\\',':','\x00'))
                or PurePosixPath(name).is_absolute() or any(p in ('','.','..') for p in parts)
                or name.casefold() in seen or info.is_dir() or info.flag_bits&1
                or stat.S_IFMT(mode) not in (0,stat.S_IFREG)):
                raise ValueError('不支持的任务 ZIP 路径或文件类型。')
            seen.add(name.casefold()); total+=info.file_size
            if info.file_size>512*1024**2 or total>4*1024**3: raise ValueError('解压内容过大。')
        if 'manifest.json' not in seen: raise ValueError('请选择 App 导出的任务 ZIP，不是确认包、结果包或源码包。')
        if z.getinfo('manifest.json').file_size>8*1024**2: raise ValueError('清单过大。')
        manifest=json.loads(z.read('manifest.json'))
        if manifest.get('format')!='splat-studio-cloud-task/1':
            raise ValueError('请选择 App 导出的任务 ZIP，不是源码 ZIP、确认包或结果包。')
        declared=set()
        for row in manifest['files']:
            name=row['path']
            if name in declared or name=='manifest.json': raise ValueError('任务清单有重复。')
            declared.add(name)
            if z.getinfo(name).file_size!=row['bytes']: raise ValueError('文件长度不匹配。')
            with z.open(name) as f: value=hashlib.file_digest(f,'sha256').hexdigest()
            if value!=row['sha256']: raise ValueError('文件校验失败：'+name)
        if declared|{'manifest.json'}!={i.filename for i in infos}: raise ValueError('任务清单与内容不一致。')
        destination.mkdir()
        for info in infos:
            target=destination/info.filename; target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(info) as src, target.open('xb') as dst: shutil.copyfileobj(src,dst)
    return manifest

run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'])
print('Python:',sys.version.split()[0])
print('初始化完成。下一步只上传一个 App 任务 ZIP。')


## 2 · 上传本次任务 ZIP

第一次上传“物品识别任务”。在 App 确认区域后，再运行本单元格上传新的“重建任务”。不覆盖此前任务。


In [ ]:
from google.colab import files
BATCH = BASE / ('task-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6])
BATCH.mkdir()
previous_cwd=Path.cwd()
try:
    os.chdir(BATCH)
    uploaded=files.upload()
finally:
    os.chdir(previous_cwd)
if len(uploaded)!=1: raise ValueError('每次只选择一个 App 导出的任务 ZIP。请重新运行本单元格。')
filename=next(iter(uploaded)); del uploaded
PACKAGE=BATCH/filename
if PACKAGE.suffix.lower()!='.zip': raise ValueError('请选择任务 ZIP。')
EXTRACTED=BATCH/'unpacked'
MANIFEST=unpack_task(PACKAGE,EXTRACTED)
RUNTIME=EXTRACTED/'runtime'
CONFIG=MANIFEST['worker_config']
PROJECT=json.loads((EXTRACTED/'project/project.json').read_text())
PHOTO_COUNT=len(PROJECT['images'])
STAGE=CONFIG.get('priority_task_stage','train')
RUN_DIR=BATCH/'run'  # 保持不存在，由 worker 创建。
_READY_TASK_ID=None
print('任务类型：','物品识别 / 等待你确认区域' if STAGE=='review' else '正式重建')
print('照片：',PHOTO_COUNT,'；质量：',CONFIG.get('mode'),'；重点：',CONFIG.get('priority_request','无'))
print('ZIP SHA256:',digest(PACKAGE))
print('本次目录：',BATCH)
if not 2<=PHOTO_COUNT<=300: raise ValueError('当前任务支持 2–300 张照片。')
if CONFIG.get('cloud_gpu')=='a100':
    gpu_names=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True)
    if 'A100' not in gpu_names: raise ValueError('App 选的是 A100，但当前 GPU 不符。切换 Colab 为 A100，或回 App 修改设备后重新导出。')
if (CONFIG.get('semantics') or CONFIG.get('priority_objects')) and not CONFIG.get('allow_model_download'):
    print('注意：任务没有允许首次下载识别模型；如果云端没有缓存，请回 App 勾选“首次使用时下载所需资源”后重新导出。')


## 3 · 设置结果备份

建议打开。Google Drive 挂载会请求你的授权；每次任务完成后复制结果，失败时保存可用日志。它不提供自动断点续训。


In [ ]:
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT=None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive/Splat-Studio')
    DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
    backup=DRIVE_ROOT/BATCH.name; backup.mkdir(exist_ok=True)
    shutil.copy2(PACKAGE,backup/PACKAGE.name)
    print('本次任务包已备份：',backup)
else:
    print('未启用 Drive 备份。任务完成后及时下载结果。')


## 4 · 安装云端环境

第一次需要下载依赖并编译扩展。采用独立 Python 环境，固定 PyTorch 2.11.0 + torchvision 0.26.0 + CUDA 12.8。


In [ ]:
INSTALL_CUDA_128_IF_NEEDED = True  # @param {type:"boolean"}
run(['apt-get','update'],log=BATCH/'apt-update.log')
run(['apt-get','install','-y','git','build-essential','ninja-build','python3-venv',
     'colmap','libgl1','libglib2.0-0'],log=BATCH/'system-setup.log')

def locate_cuda_128():
    candidates=[Path('/usr/local/cuda-12.8/bin/nvcc')]
    located=shutil.which('nvcc')
    if located: candidates.append(Path(located).resolve())
    for candidate in candidates:
        if candidate.is_file():
            version=subprocess.check_output([str(candidate),'--version'],text=True)
            if re.search(r'release\s+12\.8\b',version): return candidate.parent.parent
    return None

CUDA_ROOT=locate_cuda_128()
if CUDA_ROOT is None and INSTALL_CUDA_128_IF_NEEDED:
    # 只安装开发工具包，不更换 Colab 的 NVIDIA 驱动。
    run(['apt-get','install','-y','cuda-toolkit-12-8'],log=BATCH/'cuda-toolkit-setup.log')
    CUDA_ROOT=locate_cuda_128()
if CUDA_ROOT is None:
    raise RuntimeError('没有找到 CUDA 12.8 的 nvcc。按使用说明配置 NVIDIA 软件源/工具包；不要只看 nvidia-smi 的 CUDA Version。')
os.environ['CUDA_HOME']=str(CUDA_ROOT)
os.environ['PATH']=str(CUDA_ROOT/'bin')+os.pathsep+os.environ.get('PATH','')
os.environ['MAX_JOBS']='2'
os.environ['QT_QPA_PLATFORM']='offscreen'
os.environ['PYTHONUNBUFFERED']='1'
os.environ['HF_HOME']=str(BASE/'model-cache')
os.environ['SPLAT_GEOMETRY_HOME']=str(BASE/'geometry')
if not PY.exists(): run([sys.executable,'-m','venv',ENV_DIR],log=BATCH/'venv-setup.log')
constraints=BASE/'constraints.txt'
constraints.write_text('torch==2.11.0\ntorchvision==0.26.0\nnumpy==2.2.6\n')
os.environ['PIP_CONSTRAINT']=str(constraints)
run([PY,'-m','pip','install','--upgrade','pip','setuptools','wheel'],log=BATCH/'pip-setup.log')
run([PY,'-m','pip','install','torch==2.11.0','torchvision==0.26.0',
     '--index-url','https://download.pytorch.org/whl/cu128'],log=BATCH/'torch-setup.log')
run([PY,'-m','pip','install','-r',RUNTIME/'requirements-semantic.txt'],log=BATCH/'requirements-setup.log')
run([PY,'-c',"import torch,torchvision; assert torch.cuda.is_available(); assert torch.version.cuda=='12.8'; print('CUDA 可用：',torch.cuda.get_device_name(0)); print(torch.__version__,torchvision.__version__)"],log=BATCH/'torch-check.log')
print('Python/CUDA 环境已就绪。下一步配置训练扩展。')


## 5 · 配置训练扩展、相机恢复和按需补全

自动区分完整照片与少视角输入，并做真实导入检查。COLMAP 特征提取/匹配使用 CPU，3DGS 和语义训练使用 A100。


In [ ]:
COLMAP_WRAPPER_SOURCE = "import os,sys,subprocess\nREAL='/usr/bin/colmap'\nargs=sys.argv[1:]\ncommand=args[0] if args else ''\nchoices={\n 'feature_extractor':('FeatureExtraction.use_gpu','SiftExtraction.use_gpu'),\n 'exhaustive_matcher':('FeatureMatching.use_gpu','SiftMatching.use_gpu'),\n}\nif command in choices:\n    options=choices[command]\n    if not any(a.split('=')[0] in ['--'+o for o in options] for a in args):\n        result=subprocess.run([REAL,command,'-h'],capture_output=True,text=True)\n        help_text=result.stdout+result.stderr\n        option=next((o for o in options if o in help_text),None)\n        if option is None: raise SystemExit('无法识别 COLMAP 的 CPU 选项，请检查 COLMAP 版本。')\n        args += ['--'+option,'0']\nos.execv(REAL,[REAL]+args)\n"
# apt 的 COLMAP 通常不带 CUDA。通过独立启动脚本指定 CPU，避免无显示器环境的 OpenGL 错误。
wrapper_dir=BASE/'bin'; wrapper_dir.mkdir(exist_ok=True)
WRAPPER=wrapper_dir/'colmap'
WRAPPER.write_text('#!'+str(PY)+'\n'+COLMAP_WRAPPER_SOURCE)
WRAPPER.chmod(0o755)
os.environ['PATH']=str(wrapper_dir)+os.pathsep+str(CUDA_ROOT/'bin')+os.pathsep+str(ENV_DIR/'bin')+os.pathsep+os.environ.get('PATH','')
cache_file=BASE/'upstream-cache.json'
setup_hash=digest(RUNTIME/'scripts/setup_upstream.py')
cache=json.loads(cache_file.read_text()) if cache_file.is_file() else {}
reuse=cache.get('setup_sha256')==setup_hash and Path(cache.get('repo','/nonexistent')).joinpath('train.py').is_file()
if reuse:
    repo=Path(cache['repo'])
    # 缓存必须仍然是此版本固定的原始仓库。
    reuse=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()=='54c035f7834b564019656c3e3fcc3646292f727d'
    reuse=reuse and not subprocess.check_output(['git','-C',str(repo),'status','--porcelain','--untracked-files=no'],text=True).strip()
if not reuse:
    run([PY,RUNTIME/'scripts/setup_upstream.py','--cuda'],log=BATCH/'cuda-extensions.log')
    repo=RUNTIME/'vendor/gaussian-splatting'
    cache_file.write_text(json.dumps({'setup_sha256':setup_hash,'repo':str(repo)}))
os.environ['SPLAT_3DGS_REPO']=str(repo)
needs_geometry=(STAGE!='review' and 2<=PHOTO_COUNT<=12 and
                (CONFIG.get('sparse_completion','auto')!='none' or CONFIG.get('geometry_backend')=='dust3r'))
if needs_geometry:
    run([PY,RUNTIME/'scripts/setup_geometry.py','--root',BASE/'geometry'],log=BATCH/'geometry-setup.log')
    # 已经显式运行安装命令，并按固定 SHA 校验几何权重。
else:
    print('本任务无需安装学习式几何补全模型。')
run([PY,'-c',"import torch,diff_gaussian_rasterization,simple_knn._C,fused_ssim; from transformers import AutoModelForZeroShotObjectDetection,AutoProcessor,SamModel,SamProcessor; assert torch.cuda.is_available(); print('3DGS 扩展和物品识别组件导入成功')"],log=BATCH/'extensions-check.log')
run([PY,RUNTIME/'scripts/cloud_worker.py',PACKAGE,'--verify-only'],log=BATCH/'task-verification.log')
_READY_TASK_ID=MANIFEST['id']
print('环境与任务校验通过。任务尚未开始；运行下一单元格。')


## 6 · 执行本次任务并保存结果

识别任务结束后回 App 核对区域；正式重建任务结束后回 App 导入场景。每次运行目录不同，不会覆盖之前结果。


In [ ]:
if _READY_TASK_ID!=MANIFEST['id']: raise RuntimeError('请先完成第 5 步校验。')
if RUN_DIR.exists(): raise RuntimeError('本次运行目录已存在。先看 worker-status.json；成功则去第 7 步下载，失败则重新运行第 2 步创建新任务目录。')
stop_monitor=threading.Event()
def monitor():
    last=None
    while not stop_monitor.wait(20):
        try:
            status_path=RUN_DIR/'worker-status.json'
            if not status_path.exists(): continue
            status=json.loads(status_path.read_text())
            job_path=RUN_DIR/'data'/MANIFEST['project_id']/('job-'+MANIFEST['id']+'.json')
            job=json.loads(job_path.read_text()) if job_path.exists() else {}
            description=job.get('message') or job.get('stage') or status.get('status')
            message=(description,round(float(job.get('progress',0))*100,1))
            if message!=last:
                print(f'[{time.strftime("%H:%M:%S")}] {message[0]} · {message[1]}%',flush=True); last=message
        except (OSError,ValueError): pass
monitor_thread=threading.Thread(target=monitor,daemon=True);monitor_thread.start()
started=time.monotonic()
try:
    run([PY,RUNTIME/'scripts/cloud_worker.py',PACKAGE,'--work-dir',RUN_DIR],log=BATCH/'worker.log')
finally:
    stop_monitor.set();monitor_thread.join(timeout=2)
    print('本次 worker 耗时：',round((time.monotonic()-started)/60,2),'分钟（不含前面的环境安装）')
    if DRIVE_ROOT:
        target=DRIVE_ROOT/BATCH.name; target.mkdir(exist_ok=True)
        for path in BATCH.glob('*.log'): shutil.copy2(path,target/path.name)
        if (RUN_DIR/'worker-status.json').is_file(): shutil.copy2(RUN_DIR/'worker-status.json',target/'worker-status.json')
        for path in (RUN_DIR/'data').glob('*/job-*.json'): shutil.copy2(path,target/path.name)
        if (RUN_DIR/'priority-review.zip').is_file(): shutil.copy2(RUN_DIR/'priority-review.zip',target/'priority-review.zip')
        if (RUN_DIR/'results').is_dir(): shutil.copytree(RUN_DIR/'results',target/'results',dirs_exist_ok=True)
        print('已保存当前可用日志/结果到 Drive：',target)
state=json.loads((RUN_DIR/'worker-status.json').read_text())
print('最终状态：',state['status'])
if state['status']=='awaiting_user_confirmation':
    print('物品识别完成。下一步下载 priority-review.zip，在 App 导入并确认区域。')
elif state['status']=='completed':
    print('完整重建已完成。下一步下载 results/scene.splat.jsonl，在 App 导入。')
else:
    raise RuntimeError(state.get('error','任务尚未完成，请检查日志。'))


## 7 · 下载到 Mac

只下载对应阶段的文件。大文件可从 Google Drive 下载；不要把 priority-review.zip 当作语义成品包。


In [ ]:
from google.colab import files
state_path=RUN_DIR/'worker-status.json'
if not state_path.is_file(): raise RuntimeError('还没有执行结果，请先完成第 6 步。')
state=json.loads(state_path.read_text())
if state.get('status')=='awaiting_user_confirmation':
    files.download(str(RUN_DIR/'priority-review.zip'))
    print('回 App：导入云端重点确认包 → 检查绿色区域 → 确认所选区域用于训练 → 准备云端重建。')
    print('拿到新的重建 ZIP 后，回到本笔记本，从第 2 步开始再运行到第 7 步。')
elif state.get('status')=='completed':
    files.download(str(RUN_DIR/'results/scene.splat.jsonl'))
    print('回 App：导入已有重建 → 包含物品语义（或仅 RGB）→ 选择 scene.splat.jsonl。')
    print('建议保留整个 results 目录。Drive 备份位置：',str(DRIVE_ROOT/BATCH.name) if DRIVE_ROOT else '未开启')
else:
    raise RuntimeError('任务尚未成功，不要导入半成品。请查看 worker-status.json 和 worker.log。')


## 出错时先看这里

- `cuda-toolkit-12-8` 找不到：当前系统未配置相应 NVIDIA 软件源。按 [NVIDIA CUDA 12.8 Linux 安装文档](https://docs.nvidia.com/cuda/archive/12.8.0/cuda-installation-guide-linux/index.html) 配置与 Ubuntu 版本匹配的仓库，再重跑第 4 步。只安装 toolkit，不重装 Colab 驱动。
- GPU 不可用/不是 A100：修改 Colab 运行时，或在 App 改成实际 GPU 后重新导出。
- 不允许下载模型：回 App 勾选“首次使用时下载所需资源”，重新导出，不编辑 manifest。
- 重点没有区域：在 App 修改名称或使用手动多边形标注，重新生成确认包。
- 工作目录已存在：成功则直接运行第 7 步；失败则回第 2 步创建新目录，不删除已完成结果。
- 版本/SHA 不匹配：使用该 ZIP 自带的 runtime；不要混用不同版本源码。
- 更多问题、稀疏视角和已有模型导入见《完整使用说明.html》。

官方参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html) · [PyTorch 版本组合](https://pytorch.org/get-started/previous-versions/) · [COLMAP 安装](https://colmap.github.io/install.html) · [原始 3DGS](https://github.com/graphdeco-inria/gaussian-splatting)
